In [6]:
!pip install pandas sentence-transformers chromadb transformers accelerate torch --upgrade


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

from transformers import pipeline

# =========
# 1. Paths
# =========

DATA_PATH = "books.csv.csv"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "books_collection"

TOP_K = 10


In [8]:
# =========================
# 2. Load and clean dataset
# =========================

def load_and_clean_data(path: str) -> pd.DataFrame:

    df = pd.read_csv(path)

    needed_cols = [
        "title", "subtitle", "authors", "categories",
        "description", "average_rating", "ratings_count", "published_year"
    ]
    existing_cols = [c for c in needed_cols if c in df.columns]
    df = df[existing_cols].copy()

    # Delete rows that have no description
    df = df.dropna(subset=["description"])


    df["desc_word_count"] = df["description"].apply(
        lambda x: len(str(x).split())
    )
    df = df[df["desc_word_count"] >= 25]


    if "subtitle" in df.columns:
        df["full_title"] = df["title"].fillna("") + " - " + df["subtitle"].fillna("")
    else:
        df["full_title"] = df["title"].fillna("")


    if "categories" in df.columns:
        df["categories"] = df["categories"].fillna("Unknown")
    else:
        df["categories"] = "Unknown"

    df["text_for_embedding"] = (
        df["full_title"].fillna("") + ". " +
        df["description"].fillna("")
    )

    df = df.reset_index(drop=True)
    return df


In [9]:
# ==========================================
# 3. Initialize embedding model and ChromaDB
# ==========================================

def init_embedding_model():
    print("Loading sentence-transformers/all-MiniLM-L6-v2 ...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    return model

def init_chroma():
    client = chromadb.Client(
        Settings(
            anonymized_telemetry=False,
            persist_directory=CHROMA_DIR
        )
    )

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(COLLECTION_NAME)
    return client, collection


# ==========================================
# 4. Build vector database from the dataset
# ==========================================

def build_vector_db(df: pd.DataFrame, model, collection):
    texts = df["text_for_embedding"].tolist()
    print(f"Generating embeddings for {len(texts)} books...")


    batch_size = 256
    ids = []
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(batch_texts, show_progress_bar=True)
        batch_ids = [str(j) for j in range(i, i + len(batch_texts))]
        ids.extend(batch_ids)
        embeddings.extend(batch_emb)


    collection.add(
        ids=ids,
        embeddings=np.array(embeddings).tolist(),
        metadatas=[
            {
                "title": df.loc[int(i), "full_title"],
                "authors": df.loc[int(i), "authors"] if "authors" in df.columns else "",
                "categories": df.loc[int(i), "categories"],
                "description": df.loc[int(i), "description"],
            }
            for i in range(len(df))
        ]
    )
    print("Vector database built and stored in Chroma.")


In [26]:
# ======================================
# 5. Emotion & Category classification
# ======================================

def init_emotion_pipeline():
    print("Loading emotion model (j-hartmann/emotion-english-distilroberta-base)...")
    emo_pipe = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        return_all_scores=True
    )
    return emo_pipe

def init_zero_shot_pipeline():
    print("Loading zero-shot classification model (facebook/bart-large-mnli)...")
    zshot = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli"
    )
    return zshot

def classify_emotion(emo_pipe, text: str):
    result = emo_pipe(text)[0]
    # return the highest lable
    best = max(result, key=lambda x: x["score"])
    return best["label"], best["score"]

def classify_category(zshot, text: str, labels=None):
    if labels is None:
        labels = ["Fiction", "Non-Fiction"]
    res = zshot(text, candidate_labels=labels)
    return res["labels"][0], res["scores"][0]


# ================================
# 6. Query function (user facing)
# ================================

def semantic_search(
    query: str,
    model,
    collection,
    emo_pipe=None,
    zshot=None,
    top_k: int = TOP_K,
    filter_by_emotion: bool = False,
    filter_by_category: bool = False
):

    print(f"User query: {query}")

    # Embed the query
    q_emb = model.encode([query])[0].tolist()

    # Vector search
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k * 3
    )

    metadatas = results["metadatas"][0]
    distances = results["distances"][0]


    emotion_label = None
    category_label = None

    if emo_pipe and filter_by_emotion:
        emotion_label, emo_score = classify_emotion(emo_pipe, query)
        print(f"Detected emotion: {emotion_label} (score={emo_score:.2f})")

    if zshot and filter_by_category:
        category_label, cat_score = classify_category(zshot, query)
        print(f"Detected category: {category_label} (score={cat_score:.2f})")

    # Apply simple filtering
    filtered = []
    for meta, dist in zip(metadatas, distances):
        ok = True
        if emotion_label:

            ok = True

        if category_label:
            if "categories" in meta and category_label.lower() not in str(meta["categories"]).lower():

                pass

        filtered.append((meta, dist))

    # Sort by distance
    filtered = sorted(filtered, key=lambda x: x[1])[:top_k]

    # Print
    print("\nTop recommendations:\n")
    for i, (meta, dist) in enumerate(filtered, start=1):
        print(f"{i}. {meta.get('title', 'Unknown title')}")
        print(f"   Author(s): {meta.get('authors', 'Unknown')}")
        print(f"   Categories: {meta.get('categories', 'Unknown')}")
        print(f"   Distance: {dist:.4f}")
        desc = meta.get("description", "")
        if len(desc) > 250:
            desc = desc[:250] + "..."
        print(f"   Description: {desc}")
        print("-" * 60)

    return filtered


# ==========================
# 7. Example main workflow
# ==========================

if __name__ == "__main__":
    # 1) Load & clean data
    df = load_and_clean_data(DATA_PATH)
    print(f"Dataset after cleaning: {df.shape[0]} books")

    # 2) Initialize models and Chroma
    embed_model = init_embedding_model()
    client, collection = init_chroma()

    # 3) Build vector DB
    build_vector_db(df, embed_model, collection)

    # 4) init emotion & zero-shot
    emo_pipe = init_emotion_pipeline()
    zshot_pipe = init_zero_shot_pipeline()

    # 5) Ask user for input
    user_query = input("Enter a description of the book you are looking for:")

    # 6) Perform semantic search
    semantic_search(
        query=user_query,
        model=embed_model,
        collection=collection,
        emo_pipe=emo_pipe,
        zshot=zshot_pipe,
        top_k=5,
        filter_by_emotion=True,
        filter_by_category=True
    )

Dataset after cleaning: 5230 books
Loading sentence-transformers/all-MiniLM-L6-v2 ...
Generating embeddings for 5230 books...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Vector database built and stored in Chroma.
Loading emotion model (j-hartmann/emotion-english-distilroberta-base)...


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Loading zero-shot classification model (facebook/bart-large-mnli)...


Device set to use cuda:0


Enter a description of the book you are looking for:A suspenseful mystery novel with unexpected plot twists and a shocking ending
User query: A suspenseful mystery novel with unexpected plot twists and a shocking ending
Detected emotion: fear (score=0.97)
Detected category: Fiction (score=0.98)

Top recommendations:

1. Absolute Power - 
   Author(s): David Baldacci
   Categories: Murder
   Distance: 0.8605
   Description: 'Accomplished and amazing - one of the hottest reads around' DAILY MAIL Set in Washington D.C., this fascinating thriller of unparalleled suspense dares to explore an unthinkable abuse of power and criminal conspiracy: a vicious murder involving the ...
------------------------------------------------------------
2. Thriller - 
   Author(s): James Patterson
   Categories: Fiction
   Distance: 0.8823
   Description: A collection of thirty tales of suspense features contributions from Heather Graham, Lincoln Child, Denise Hamilton, Michael Palmer, Douglas Preston, Alex

In [27]:
import random
import math
import numpy as np

def evaluate_recommender_system(
    df,
    model,
    collection,
    samples=50,
    top_k=5
):
    """
    Unified evaluation:
    1) Category-based relevance evaluation -> Precision@k
    2) Self-retrieval evaluation -> Hit@k and nDCG@k
    """

    # ===============================
    # Part 1: Category-based Precision@k
    # ===============================
    subset = df.sample(samples)
    precision_scores = []

    for _, row in subset.iterrows():
        true_category = row.get("categories", "")
        if not isinstance(true_category, str) or not true_category.strip():
            continue

        query_text = row.get("description", "")
        if not isinstance(query_text, str) or not query_text.strip():
            continue

        q_emb = model.encode([query_text])[0].tolist()

        results = collection.query(
            query_embeddings=[q_emb],
            n_results=top_k
        )

        metadatas = results.get("metadatas", [[]])[0]
        if not metadatas:
            continue

        retrieved_relevant = 0
        for meta in metadatas:
            book_cat = meta.get("categories", "")
            if isinstance(book_cat, str) and true_category.lower() in book_cat.lower():
                retrieved_relevant += 1

        precision = retrieved_relevant / top_k
        precision_scores.append(precision)

    avg_precision = float(np.mean(precision_scores)) if precision_scores else 0.0

    # ===============================
    # Part 2: Self-retrieval (Hit@k, nDCG@k)
    # ===============================
    indices = random.sample(range(len(df)), samples)
    hits = 0
    ndcg_total = 0.0

    for i in indices:
        query_text = df.loc[i, "text_for_embedding"]
        true_id = str(i)

        q_emb = model.encode([query_text])[0].tolist()

        res = collection.query(
            query_embeddings=[q_emb],
            n_results=top_k
        )

        retrieved_ids = res["ids"][0]

        if true_id in retrieved_ids:
            hits += 1
            rank = retrieved_ids.index(true_id) + 1
            dcg = 1 / math.log2(rank + 1)
        else:
            dcg = 0.0

        ndcg_total += dcg  # IDCG = 1 in self-retrieval

    hit_at_k = hits / samples
    avg_ndcg = ndcg_total / samples

    # ===============================
    # Print results
    # ===============================
    print("=== Evaluation Results ===")
    print(f"Samples used: {samples}")
    print(f"Precision@{top_k}: {avg_precision:.3f}")
    print(f"Hit@{top_k}: {hit_at_k:.3f}")
    print(f"nDCG@{top_k}: {avg_ndcg:.3f}")

    return {
        "Precision@k": avg_precision,
        "Hit@k": hit_at_k,
        "nDCG@k": avg_ndcg
    }


# Run evaluation
evaluate_recommender_system(df, embed_model, collection, samples=50, top_k=5)


=== Evaluation Results ===
Samples used: 50
Precision@5: 0.560
Hit@5: 1.000
nDCG@5: 1.000


{'Precision@k': 0.56, 'Hit@k': 1.0, 'nDCG@k': 1.0}

In [28]:
!pip install gradio -q


In [29]:
import gradio as gr

def recommend_ui(query, top_k):

    if not query or query.strip() == "":
        return "Please enter a description for the book you want:"


    results = semantic_search(
        query=query,
        model=embed_model,
        collection=collection,
        emo_pipe=None,
        zshot=None,
        top_k=top_k,
        filter_by_emotion=False,
        filter_by_category=False
    )

    lines = []
    for i, (meta, dist) in enumerate(results, start=1):
        title = meta.get("title", "Unknown title")
        authors = meta.get("authors", "Unknown")
        categories = meta.get("categories", "Unknown")
        desc = meta.get("description", "")
        if len(desc) > 250:
            desc = desc[:250] + "..."

        lines.append(
            f"{i}. {title}\n"
            f"   Author(s): {authors}\n"
            f"   Categories: {categories}\n"
            f"   Distance: {dist:.4f}\n"
            f"   Description: {desc}\n"
            + "-" * 60
        )

    return "\n".join(lines) if lines else "No recommendations foundو Please try providing a more detailed or clearer query."


In [30]:
demo = gr.Interface(
    fn=recommend_ui,
    inputs=[
        gr.Textbox(
            lines=3,
            label="Enter a description of the book you are looking for",
            placeholder="e.g., a sad romantic story about losing someone you love"
        ),
        gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Number of recommendations (Top-K)"
        )
    ],
    outputs=gr.Textbox(
        lines=20,
        label="Recommended Books"
    ),
    title="Semantic Book Recommender",
    description="Enter a natural-language description and get semantically similar book recommendations."
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a3ec4942f7aab4059.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
